In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
import cv2

from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
CLASS_MAP = {
    "mat_bo_phan": 1,
    "rach": 2,
    "mop_lom": 3,
    "tray_son": 4
}

In [5]:
class DamageDataset(Dataset):
    def __init__(self, image_dir, annotation_path):
        self.image_dir = image_dir

        with open(annotation_path, 'r') as f:
            self.annotations = json.load(f)

        if isinstance(self.annotations, list):
            self.image_files = self.annotations
            self.is_list_format = True
        else:
            self.image_files = list(self.annotations.keys())
            self.is_list_format = False

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        if self.is_list_format:
            data = self.image_files[idx]
            img_name = data["image_name"]
            boxes = data["boxes"]
            labels = data["labels"]
        else:
            img_name = self.image_files[idx]
            data = self.annotations[img_name]

            boxes = []
            labels = []

            for region in data["regions"]:
                if region["class"] not in CLASS_MAP:
                    continue

                xs = region["all_x"]
                ys = region["all_y"]

                xmin, ymin = min(xs), min(ys)
                xmax, ymax = max(xs), max(ys)

                if xmax <= xmin or ymax <= ymin:
                    continue

                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(CLASS_MAP[region["class"]])

        img_path = os.path.join(self.image_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        image = torch.as_tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return image, target

In [6]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [7]:
NUM_CLASSES = 5

model = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=600, max_size=800)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()

print("Best model loaded.")

C:\Users\HP\AppData\Local\Temp\ipykernel_28520\1338890875.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pth", map_location

Best model loaded.


In [8]:
VAL_IMG_DIR = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\val_images"
VAL_JSON = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\0Val_via_annos.json"

val_dataset = DamageDataset(VAL_IMG_DIR, VAL_JSON)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print("Validation samples:", len(val_dataset))

Validation samples: 2324


In [9]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union = area1 + area2 - inter_area

    if union == 0:
        return 0

    return inter_area / union

In [10]:
iou_threshold = 0.1
score_threshold = 0.6

TP, FP, FN = 0, 0, 0

with torch.no_grad():
    for images, targets in val_loader:

        images = [img.to(device) for img in images]
        predictions = model(images)

        for pred, target in zip(predictions, targets):

            gt_boxes = target["boxes"].cpu().numpy()
            gt_labels = target["labels"].cpu().numpy()

            pred_boxes = pred["boxes"].cpu().numpy()
            pred_labels = pred["labels"].cpu().numpy()
            pred_scores = pred["scores"].cpu().numpy()

            valid_idx = pred_scores > score_threshold
            pred_boxes = pred_boxes[valid_idx]
            pred_labels = pred_labels[valid_idx]

            matched_gt = set()
            matched_pred = set()

            for i, (p_box, p_label) in enumerate(zip(pred_boxes, pred_labels)):
                best_iou = 0
                best_gt_idx = -1

                for j, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
                    if j in matched_gt:
                        continue
                    if p_label != gt_label:
                        continue

                    iou = compute_iou(p_box, gt_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = j

                if best_iou >= iou_threshold:
                    TP += 1
                    matched_gt.add(best_gt_idx)
                    matched_pred.add(i)

            FP += len(pred_boxes) - len(matched_pred)
            FN += len(gt_boxes) - len(matched_gt)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", round(precision * 100, 2), "%")
print("Recall:", round(recall * 100, 2), "%")
print("F1 Score:", round(f1 * 100, 2), "%")

Precision: 39.52 %
Recall: 40.94 %
F1 Score: 40.22 %
